## Measure Interior/Exterior — Depth + Planes + Semantic Masks → Approx. Real‑World Measurements

This notebook implements a **practical, Python-first** pipeline to estimate depth, build a point cloud, detect structural elements (walls/floor/ceiling/railings/mouldings/baseboards) and common openings (windows/doors), and compute **approximate metric measurements** (meters): distances, heights, widths, and areas.

### What you need
- **Single image mode (monocular)**: 1 RGB image + **one reference measurement** (e.g., known door height, wall segment length). This resolves the **scale ambiguity** in monocular depth.
- **Multi-view mode (COLMAP)**: 5–30 photos of the same scene from different viewpoints. Metric scale still needs a reference unless you have a known baseline/sensor metadata.
- **RGB-D / SLAM mode**: RGB-D frames (e.g., RealSense) or a SLAM-produced trajectory + depth. This is preferred for **metric** scanning.

### Accuracy expectations (important)
- Monocular depth is **not metric by default**; it is typically correct **up to an unknown global scale**.
- Measurements here are **estimates**. Expect errors from: reflective/transparent surfaces, motion blur, textureless walls, occlusions, inaccurate intrinsics, and imperfect masks.
- For real projects: capture multiple views, use known camera intrinsics, and validate with a tape measure.

### Privacy / legal / safety
- Only process images you have rights to use.
- Interior photos may contain sensitive information; keep everything local.
- Do not use this notebook as the sole basis for safety-critical decisions.

### OSS components used
- Depth: [Depth Anything V2](https://github.com/DepthAnything/Depth-Anything-V2) (default) + optional fallbacks.
- Geometry: [Open3D](http://www.open3d.org/) (point clouds, plane fitting, measurements).
- Semantics (fallback): Hugging Face `transformers` segmentation models.
- Optional: COLMAP for SfM/MVS; Detectron2/PlaneRCNN/RTAB-Map/ORB-SLAM3 via external installs.


## 1) Install (pip-first) + optional system installs

This notebook prefers **pip installs**. Some components (COLMAP / SLAM toolchains / Detectron2) are heavy and may require system tools.

If you run into dependency issues, use a fresh virtualenv/conda env.


In [1]:
# If you're in VS Code / Jupyter, run this once.
# It installs Python deps from requirements.txt in this folder.

%pip install -r requirements.txt

# Torch note:
# - NVIDIA GPU: follow https://pytorch.org/get-started/locally/ for the correct CUDA build.
# - macOS: MPS will be used automatically when available.

import sys, platform, shutil, subprocess
print("Python:", sys.version)
print("Platform:", platform.platform())
print("colmap available:", bool(shutil.which("colmap")))
print("ffmpeg available:", bool(shutil.which("ffmpeg")))

# ---- Optional system installs ----
# COLMAP:
#   macOS:  brew install colmap
#   Ubuntu: sudo apt-get update && sudo apt-get install -y colmap
# If COLMAP isn't available, you can still use Single-image mode.

# Detectron2 (optional): install depends on OS/Python/CUDA; see:
#   https://github.com/facebookresearch/detectron2/blob/main/INSTALL.md

# PlaneRCNN / ORB-SLAM3 / RTAB-Map:
#   typically require source builds; this notebook provides command cells + Python fallbacks.


  Using cached numpy-2.4.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached opencv_python-4.13.0.90-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
  Using cached matplotlib-3.10.8-cp314-cp314-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached pandas-3.0.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.26.0 Requires-Python >=3.9,<3.13; 1.26.1 Requires-Python >=3.9,<3.13
ERROR: Could not find a version that satisfies the requirement open3d>=0.18 (from versions: none)
ERROR: No matching distribution found for open3d>=0.18
Note: you may need to restart the kernel to use updated packages.
Python: 3.14.2 (main, Dec  5 2025, 16:49:16) [Clang 17.0.0 (clang-1700.4.4.1)]
Platform: macOS-15.6.1-arm64-arm-64bit-Mac

### Docker fallback (optional)

If your local Python environment struggles with native packages (Open3D / PyTorch / etc.), run in Docker.

- Base idea: mount this folder into a container with Python + PyTorch + Open3D.
- Example (Linux + NVIDIA): use an official `pytorch/pytorch` CUDA image and install deps.

Example commands (edit as needed):

```bash
docker run --rm -it \
  -p 8888:8888 \
  -v "$(pwd)":/workspace \
  -w /workspace \
  pytorch/pytorch:2.3.1-cuda12.1-cudnn8-runtime \
  bash -lc "pip install -r requirements.txt && jupyter lab --ip=0.0.0.0 --no-browser --allow-root"
```

On macOS, Docker won’t give you GPU acceleration, but it can still help with dependency isolation.


## 2) Imports + paths

We’ll use `Depth-Anything-V2/` (already cloned into this folder) by adding it to `sys.path`.


In [2]:
import os
import sys
import json
import math
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch

# Optional imports (handled gracefully)
try:
    import open3d as o3d
except Exception as e:
    o3d = None
    print("Open3D import failed:", e)

try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception as e:
    widgets = None
    print("ipywidgets import failed:", e)

ROOT = Path(".").resolve()
DA2_ROOT = ROOT / "Depth-Anything-V2"
CHECKPOINTS_DIR = DA2_ROOT / "checkpoints"
SAMPLE_DIR = ROOT / "sample_data"

assert DA2_ROOT.exists(), f"Expected {DA2_ROOT} to exist. Re-clone Depth-Anything-V2 into this folder."

sys.path.insert(0, str(DA2_ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("SAMPLE_DIR:", SAMPLE_DIR)

def imread_rgb(path: str | Path) -> np.ndarray:
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def show_img(img_rgb: np.ndarray, title: str = "", figsize=(10, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis("off")
    plt.show()


ModuleNotFoundError: No module named 'numpy'

## 3) Mode selector UI

Choose one of:
- **single**: one RGB image (needs scale calibration)
- **multiview**: folder of images (COLMAP; still needs scale unless you have a baseline)
- **rgbd**: RGB-D frames or RGB+depth inputs (metric if depth is metric)


In [ ]:
DEFAULT_IMAGE = str((ROOT / "images" / "test.jpeg").resolve()) if (ROOT / "images" / "test.jpeg").exists() else str((SAMPLE_DIR / "demo01.jpg").resolve())
DEFAULT_IMAGE_FOLDER = str(SAMPLE_DIR.resolve())

@dataclass
class RunConfig:
    mode: str = "single"  # single | multiview | rgbd
    input_path: str = DEFAULT_IMAGE
    input_folder: str = DEFAULT_IMAGE_FOLDER
    depth_encoder: str = "vits"  # vits|vitb|vitl
    depth_input_size: int = 518
    use_semantic_segmentation: bool = True
    use_maskrcnn_instances: bool = True
    # Intrinsics (can override); if None we estimate from FOV.
    fx: float | None = None
    fy: float | None = None
    cx: float | None = None
    cy: float | None = None
    fov_degrees: float = 60.0
    # Calibration (single-image): known length in meters between two clicked points.
    known_length_m: float = 1.0

cfg = RunConfig()

if widgets is not None:
    w_mode = widgets.Dropdown(options=["single", "multiview", "rgbd"], value=cfg.mode, description="Mode")
    w_input = widgets.Text(value=cfg.input_path, description="Input image")
    w_folder = widgets.Text(value=cfg.input_folder, description="Input folder")
    w_encoder = widgets.Dropdown(options=["vits", "vitb", "vitl"], value=cfg.depth_encoder, description="DA2 encoder")
    w_size = widgets.IntSlider(value=cfg.depth_input_size, min=256, max=1024, step=14, description="Input size")
    w_semseg = widgets.Checkbox(value=cfg.use_semantic_segmentation, description="Semantic seg")
    w_inst = widgets.Checkbox(value=cfg.use_maskrcnn_instances, description="Mask R-CNN")
    w_known = widgets.FloatText(value=cfg.known_length_m, description="Known length (m)")

    display(widgets.VBox([
        w_mode,
        widgets.HTML("<b>Single-image:</b> set Input image. <b>Multi-view:</b> set Input folder. <b>RGB-D:</b> provide folder with rgb/ + depth/ subfolders."),
        w_input,
        w_folder,
        widgets.HBox([w_encoder, w_size]),
        widgets.HBox([w_semseg, w_inst]),
        w_known,
    ]))

    def _apply(_=None):
        cfg.mode = w_mode.value
        cfg.input_path = w_input.value
        cfg.input_folder = w_folder.value
        cfg.depth_encoder = w_encoder.value
        cfg.depth_input_size = int(w_size.value)
        cfg.use_semantic_segmentation = bool(w_semseg.value)
        cfg.use_maskrcnn_instances = bool(w_inst.value)
        cfg.known_length_m = float(w_known.value)
        print("Updated cfg:", cfg)

    btn = widgets.Button(description="Apply config")
    btn.on_click(_apply)
    display(btn)
else:
    print("ipywidgets not available; edit cfg in this cell.")
    print(cfg)


## 4) Depth model (Depth Anything V2) — download + load

We default to `vits` (small) for reliability and speed. You can switch to `vitl` for better quality (larger download).


In [ ]:
from huggingface_hub import hf_hub_download

DA2_MODEL_CONFIGS = {
    "vits": {"encoder": "vits", "features": 64, "out_channels": [48, 96, 192, 384]},
    "vitb": {"encoder": "vitb", "features": 128, "out_channels": [96, 192, 384, 768]},
    "vitl": {"encoder": "vitl", "features": 256, "out_channels": [256, 512, 1024, 1024]},
}

DA2_HF = {
    "vits": ("depth-anything/Depth-Anything-V2-Small", "depth_anything_v2_vits.pth"),
    "vitb": ("depth-anything/Depth-Anything-V2-Base", "depth_anything_v2_vitb.pth"),
    "vitl": ("depth-anything/Depth-Anything-V2-Large", "depth_anything_v2_vitl.pth"),
}

def ensure_da2_checkpoint(encoder: str) -> Path:
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    repo_id, filename = DA2_HF[encoder]
    local_path = CHECKPOINTS_DIR / filename
    if local_path.exists() and local_path.stat().st_size > 10_000_000:
        return local_path

    print(f"Downloading {encoder} checkpoint to {local_path} ...")
    downloaded = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(CHECKPOINTS_DIR), local_dir_use_symlinks=False)
    return Path(downloaded)

# Depth Anything V2 local import
from depth_anything_v2.dpt import DepthAnythingV2

def load_depth_anything_v2(encoder: str, device: str = DEVICE) -> DepthAnythingV2:
    ckpt = ensure_da2_checkpoint(encoder)
    model = DepthAnythingV2(**DA2_MODEL_CONFIGS[encoder])
    state = torch.load(str(ckpt), map_location="cpu")
    model.load_state_dict(state)
    model = model.to(device).eval()
    return model

# Lazy singleton cache
_DA2_MODEL = None
_DA2_ENCODER = None

def get_depth_model() -> DepthAnythingV2:
    global _DA2_MODEL, _DA2_ENCODER
    if _DA2_MODEL is None or _DA2_ENCODER != cfg.depth_encoder:
        _DA2_MODEL = load_depth_anything_v2(cfg.depth_encoder)
        _DA2_ENCODER = cfg.depth_encoder
        print("Loaded Depth Anything V2:", _DA2_ENCODER)
    return _DA2_MODEL


## 5) Input reading (single image / folder / video)

This notebook accepts:
- a single image file (`.jpg/.png`)
- a folder of images (multi-view)
- a video file (for RGB-D/SLAM workflows, we usually need depth too)


In [ ]:
from typing import List, Tuple, Dict, Optional

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def list_images(folder: str | Path) -> List[Path]:
    folder = Path(folder)
    imgs = [p for p in folder.glob("**/*") if p.suffix.lower() in IMAGE_EXTS]
    return sorted(imgs)

def read_single_image(path: str | Path) -> np.ndarray:
    img = imread_rgb(path)
    return img

def read_image_folder(folder: str | Path, limit: int = 200) -> List[np.ndarray]:
    paths = list_images(folder)[:limit]
    if not paths:
        raise FileNotFoundError(f"No images in folder: {folder}")
    return [imread_rgb(p) for p in paths]

# Optional: video frame extraction (RGB only)
def read_video_frames(video_path: str | Path, max_frames: int = 60, stride: int = 5) -> List[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")
    frames = []
    idx = 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        if idx % stride == 0:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            frames.append(rgb)
            if len(frames) >= max_frames:
                break
        idx += 1
    cap.release()
    return frames

print("Example single image:", cfg.input_path)
print("Example folder:", cfg.input_folder)


## 6) Depth estimation (Depth Anything V2) + visualization

- Output is **relative depth** (scale-ambiguous) in single-image mode.
- In multi-view / RGB-D modes, scale is established via reconstruction or sensor.

You can increase `cfg.depth_input_size` for more detail (slower).


In [ ]:
def infer_depth_da2(img_rgb: np.ndarray, input_size: int) -> np.ndarray:
    """Returns HxW float depth (relative)"""
    model = get_depth_model()
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    with torch.no_grad():
        depth = model.infer_image(bgr, input_size=input_size)
    return depth

def depth_to_vis(depth: np.ndarray) -> np.ndarray:
    d = depth.astype(np.float32)
    d = (d - np.nanmin(d)) / (np.nanmax(d) - np.nanmin(d) + 1e-8)
    cmap = plt.colormaps.get_cmap("Spectral_r")
    rgb = (cmap((d * 255).astype(np.uint8))[:, :, :3] * 255).astype(np.uint8)
    return rgb

# Quick test on current cfg (single image)
if cfg.mode == "single":
    img = read_single_image(cfg.input_path)
    show_img(img, "Input")
    depth = infer_depth_da2(img, cfg.depth_input_size)
    show_img(depth_to_vis(depth), f"Depth (relative) — encoder={cfg.depth_encoder}")
    print("Depth range:", float(depth.min()), float(depth.max()))


## 7) Camera intrinsics + depth → point cloud

We need intrinsics to convert depth to a point cloud.

### Defaults
If you don’t know intrinsics, we estimate focal length from an assumed horizontal FOV (default 60°):
- \( f_x \approx \frac{W}{2\tan(\text{FOV}/2)} \)
- \( c_x = W/2, c_y = H/2 \)

You can override `cfg.fx/fy/cx/cy`.


In [ ]:
@dataclass
class Intrinsics:
    fx: float
    fy: float
    cx: float
    cy: float

def estimate_intrinsics_from_fov(img_shape: Tuple[int,int,int], fov_degrees: float) -> Intrinsics:
    h, w = img_shape[:2]
    fov = math.radians(fov_degrees)
    fx = (w / 2.0) / math.tan(fov / 2.0)
    fy = fx
    cx = w / 2.0
    cy = h / 2.0
    return Intrinsics(fx=fx, fy=fy, cx=cx, cy=cy)

def get_intrinsics(img: np.ndarray) -> Intrinsics:
    est = estimate_intrinsics_from_fov(img.shape, cfg.fov_degrees)
    fx = cfg.fx if cfg.fx is not None else est.fx
    fy = cfg.fy if cfg.fy is not None else est.fy
    cx = cfg.cx if cfg.cx is not None else est.cx
    cy = cfg.cy if cfg.cy is not None else est.cy
    return Intrinsics(fx=float(fx), fy=float(fy), cx=float(cx), cy=float(cy))

def depth_to_pointcloud_o3d(img_rgb: np.ndarray, depth: np.ndarray, K: Intrinsics, depth_scale: float = 1.0, stride: int = 2):
    """Create point cloud from depth (depth units are arbitrary unless calibrated).

    depth_scale multiplies the raw depth values; in monocular mode we set it after calibration.
    """
    if o3d is None:
        raise RuntimeError("Open3D not available. Install open3d and restart kernel.")

    h, w = depth.shape
    ys = np.arange(0, h, stride)
    xs = np.arange(0, w, stride)
    xv, yv = np.meshgrid(xs, ys)
    z = depth[yv, xv].astype(np.float32) * float(depth_scale)

    # Backproject
    x = (xv.astype(np.float32) - K.cx) * z / K.fx
    y = (yv.astype(np.float32) - K.cy) * z / K.fy

    pts = np.stack([x, y, z], axis=-1).reshape(-1, 3)
    cols = img_rgb[yv, xv].reshape(-1, 3).astype(np.float32) / 255.0

    # Filter invalid
    mask = np.isfinite(pts).all(axis=1) & (pts[:, 2] > 0)
    pts = pts[mask]
    cols = cols[mask]

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    pcd.colors = o3d.utility.Vector3dVector(cols)
    return pcd

# Quick point cloud preview (unscaled in monocular mode)
if cfg.mode == "single" and o3d is not None:
    img = read_single_image(cfg.input_path)
    depth = infer_depth_da2(img, cfg.depth_input_size)
    K = get_intrinsics(img)
    pcd = depth_to_pointcloud_o3d(img, depth, K, depth_scale=1.0, stride=3)
    print("PCD points:", np.asarray(pcd.points).shape)
    # Use Open3D viewer (opens a window)
    # o3d.visualization.draw_geometries([pcd])


## 8) Semantic detection (walls/windows/doors/ceiling/fixtures)

We support two layers:

- **Semantic segmentation (recommended)**: uses a Hugging Face segmentation model to get masks for classes like *wall / floor / ceiling / door / window* (availability depends on model).
- **Instance detection (optional)**: torchvision Mask R-CNN (COCO) helps with furniture-like items, but COCO does **not** include doors/windows.

If Detectron2 is installed, you can optionally swap in Detectron2 models.


In [ ]:
# ---- Semantic segmentation (transformers fallback) ----
# We use an ADE20K semantic model. It includes classes like wall/floor/ceiling/window/door.
# On first run it downloads weights.

from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

# A widely used ADE20K semantic segmentation checkpoint:
SEMSEG_MODEL_ID = "nvidia/segformer-b0-finetuned-ade-512-512"

_semseg_processor = None
_semseg_model = None
_semseg_id2label = None

def get_semseg():
    global _semseg_processor, _semseg_model, _semseg_id2label
    if _semseg_model is None:
        _semseg_processor = AutoImageProcessor.from_pretrained(SEMSEG_MODEL_ID)
        _semseg_model = AutoModelForSemanticSegmentation.from_pretrained(SEMSEG_MODEL_ID).to(DEVICE).eval()
        _semseg_id2label = _semseg_model.config.id2label
        print("Loaded semseg:", SEMSEG_MODEL_ID)
    return _semseg_processor, _semseg_model, _semseg_id2label

TARGET_LABELS = {
    # These labels depend on ADE mapping; we match by substring.
    "wall": ["wall"],
    "floor": ["floor"],
    "ceiling": ["ceiling"],
    "window": ["window"],
    "door": ["door"],
    "railing": ["railing", "banister"],
    "stairs": ["stairs", "stair"],
}

def semseg_masks(img_rgb: np.ndarray) -> Dict[str, np.ndarray]:
    processor, model, id2label = get_semseg()
    inputs = processor(images=img_rgb, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # [B, C, H', W']

    up = torch.nn.functional.interpolate(logits, size=img_rgb.shape[:2], mode="bilinear", align_corners=False)
    pred = up.argmax(dim=1)[0].detach().cpu().numpy().astype(np.int32)  # HxW label id

    # Build masks by string matching labels
    masks: Dict[str, np.ndarray] = {}
    label_names = {i: id2label[i].lower() for i in id2label}

    for key, needles in TARGET_LABELS.items():
        ids = [i for i, name in label_names.items() if any(n in name for n in needles)]
        if not ids:
            masks[key] = np.zeros(pred.shape, dtype=bool)
            continue
        m = np.isin(pred, ids)
        masks[key] = m

    return masks

def overlay_masks(img_rgb: np.ndarray, masks: Dict[str, np.ndarray], alpha: float = 0.45) -> np.ndarray:
    out = img_rgb.copy().astype(np.float32)
    colors = {
        "wall": (255, 0, 0),
        "floor": (0, 255, 0),
        "ceiling": (0, 0, 255),
        "window": (255, 255, 0),
        "door": (255, 0, 255),
        "railing": (0, 255, 255),
        "stairs": (255, 128, 0),
    }
    for k, m in masks.items():
        if m is None:
            continue
        c = np.array(colors.get(k, (200, 200, 200)), dtype=np.float32)
        out[m] = (1 - alpha) * out[m] + alpha * c
    return out.astype(np.uint8)

# Optional: Mask R-CNN for furniture-ish instances
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.transforms.functional import to_tensor

_maskrcnn = None

def get_maskrcnn():
    global _maskrcnn
    if _maskrcnn is None:
        _maskrcnn = maskrcnn_resnet50_fpn(weights="DEFAULT").to(DEVICE).eval()
        print("Loaded torchvision Mask R-CNN (COCO)")
    return _maskrcnn

# COCO class names (short list for common furniture)
COCO_KEEP = {
    "chair", "couch", "bed", "dining table", "tv", "laptop", "backpack", "refrigerator", "microwave", "toaster", "sink", "book", "vase",
}

# From torchvision docs
COCO_CATEGORIES = [
    "__background__", "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light", "fire hydrant",
    "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack",
    "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard",
    "surfboard", "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
    "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote",
    "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
]

def maskrcnn_instances(img_rgb: np.ndarray, score_thresh: float = 0.6) -> List[Dict]:
    model = get_maskrcnn()
    x = to_tensor(img_rgb).to(DEVICE)
    with torch.no_grad():
        pred = model([x])[0]
    out = []
    for i in range(len(pred["scores"])):
        s = float(pred["scores"][i].detach().cpu())
        if s < score_thresh:
            continue
        cid = int(pred["labels"][i].detach().cpu())
        name = COCO_CATEGORIES[cid] if cid < len(COCO_CATEGORIES) else str(cid)
        if name not in COCO_KEEP:
            continue
        mask = (pred["masks"][i, 0].detach().cpu().numpy() > 0.5)
        box = pred["boxes"][i].detach().cpu().numpy().tolist()
        out.append({"label": name, "score": s, "mask": mask, "box": box})
    return out

# Quick semseg preview
if cfg.mode == "single" and cfg.use_semantic_segmentation:
    img = read_single_image(cfg.input_path)
    masks = semseg_masks(img)
    show_img(overlay_masks(img, masks), "Semantic overlay (SegFormer ADE20K)")
    print({k: int(v.sum()) for k, v in masks.items()})


## 9) Plane detection + moulding/baseboard/rail heuristics

Instead of PlaneRCNN (heavy), we do **geometric plane fitting** with Open3D RANSAC.

- We fit multiple planes iteratively.
- We label planes as *floor/ceiling* (near-horizontal normals) vs *walls* (near-vertical normals).
- We then estimate plane extents.

For baseboards/crown moulding/rails: we use edge detection + morphology near wall boundaries as a heuristic (works best when well-lit and high-contrast).


In [ ]:
def fit_planes_ransac(pcd, max_planes: int = 5, distance_threshold: float = 0.02, ransac_n: int = 3, num_iter: int = 2000):
    """Iteratively extract planes.

    Returns list of dicts: {plane_model, inlier_idx, pcd_inliers, pcd_rest}
    """
    if o3d is None:
        raise RuntimeError("Open3D not available")

    planes = []
    remaining = pcd
    for _ in range(max_planes):
        if len(remaining.points) < 500:
            break
        plane_model, inliers = remaining.segment_plane(
            distance_threshold=distance_threshold,
            ransac_n=ransac_n,
            num_iterations=num_iter,
        )
        inliers = np.array(inliers, dtype=np.int64)
        if inliers.size < 500:
            break
        p_in = remaining.select_by_index(inliers)
        p_out = remaining.select_by_index(inliers, invert=True)
        planes.append({"plane_model": plane_model, "inliers": inliers, "pcd_in": p_in})
        remaining = p_out
    return planes, remaining

def plane_normal(plane_model) -> np.ndarray:
    a, b, c, d = plane_model
    n = np.array([a, b, c], dtype=np.float32)
    n = n / (np.linalg.norm(n) + 1e-8)
    return n

def classify_plane(plane_model) -> str:
    n = plane_normal(plane_model)
    # z axis points forward in our camera coords (approx). 'vertical' axis is y.
    # Here we treat y as vertical, so floor/ceiling normals are roughly +/-y.
    y = np.array([0, 1, 0], dtype=np.float32)
    cos = float(abs(np.dot(n, y)))
    if cos > 0.8:
        return "horizontal"  # floor/ceiling
    return "vertical"  # walls etc

def plane_extent(pcd_in) -> Dict:
    pts = np.asarray(pcd_in.points)
    if pts.shape[0] == 0:
        return {"center": [0,0,0], "extent": [0,0,0]}
    obb = pcd_in.get_oriented_bounding_box()
    return {"center": obb.center.tolist(), "extent": obb.extent.tolist()}

def detect_moulding_baseboard_heuristic(img_rgb: np.ndarray, wall_mask: Optional[np.ndarray] = None) -> Dict[str, np.ndarray]:
    """Heuristic 2D detection (not metric): edges + morphology.

    Returns masks for baseboard/crown/rail candidates.
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 160)

    # Dilate edges slightly
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    ed = cv2.dilate(edges, k, iterations=1)

    # Baseboard tends to be near bottom of walls: focus on bottom 25%
    h, w = gray.shape
    base_roi = np.zeros_like(ed, dtype=np.uint8)
    base_roi[int(h*0.70):, :] = 255

    crown_roi = np.zeros_like(ed, dtype=np.uint8)
    crown_roi[:int(h*0.20), :] = 255

    rail_roi = np.zeros_like(ed, dtype=np.uint8)
    rail_roi[int(h*0.30):int(h*0.70), :] = 255

    if wall_mask is not None:
        wm = wall_mask.astype(np.uint8) * 255
        base = (ed & base_roi & wm) > 0
        crown = (ed & crown_roi & wm) > 0
        rail = (ed & rail_roi & wm) > 0
    else:
        base = (ed & base_roi) > 0
        crown = (ed & crown_roi) > 0
        rail = (ed & rail_roi) > 0

    return {"baseboard": base, "crown_moulding": crown, "railing_like": rail}


## 10) Metric calibration (single image)

Monocular depth is scale-ambiguous. We provide 3 calibration options:

1. **Known length between two image points** (recommended): click 2 points on the image, enter the real length in meters.
2. **Known object height** (e.g., door): use door mask and a known height.
3. **Multi-view / RGB-D**: scale comes from reconstruction or sensor.

We compute a single global scale factor \(s\) so that:

\[ \|X_2 - X_1\|_{meters} = s \cdot \|X_2 - X_1\|_{depth\ units} \]

This is an approximation (works best when the reference object is in the same depth range as your target measurements).


In [ ]:
def backproject_pixel(u: float, v: float, depth_value: float, K: Intrinsics) -> np.ndarray:
    """Backproject one pixel to 3D (depth units)."""
    z = float(depth_value)
    x = (u - K.cx) * z / K.fx
    y = (v - K.cy) * z / K.fy
    return np.array([x, y, z], dtype=np.float32)

def click_two_points(img_rgb: np.ndarray, title: str = "Click 2 points, then close the figure") -> List[Tuple[float,float]]:
    """Returns [(u1,v1),(u2,v2)] in pixel coords.

    Tip: for reliable clicking in Jupyter:
      - run `%matplotlib widget` (if available), or
      - use VS Code notebook interactive backend.
    """
    plt.figure(figsize=(10,6))
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis('off')
    pts = plt.ginput(2, timeout=0)
    plt.close()
    if len(pts) != 2:
        raise RuntimeError("Expected exactly 2 clicks")
    return [(float(x), float(y)) for x,y in pts]

def estimate_scale_from_two_points(img_rgb: np.ndarray, depth: np.ndarray, K: Intrinsics, known_length_m: float) -> Dict:
    pts = click_two_points(img_rgb, title=f"Click 2 points with known distance = {known_length_m}m")
    (u1,v1),(u2,v2) = pts
    d1 = float(depth[int(round(v1)), int(round(u1))])
    d2 = float(depth[int(round(v2)), int(round(u2))])
    X1 = backproject_pixel(u1, v1, d1, K)
    X2 = backproject_pixel(u2, v2, d2, K)
    dist_units = float(np.linalg.norm(X2 - X1))
    scale = float(known_length_m / (dist_units + 1e-8))
    return {
        "method": "two_points",
        "points_px": pts,
        "known_length_m": float(known_length_m),
        "dist_units": dist_units,
        "scale_to_m": scale,
    }

# Run calibration only when needed
calib = None
if cfg.mode == "single":
    img = read_single_image(cfg.input_path)
    depth = infer_depth_da2(img, cfg.depth_input_size)
    K = get_intrinsics(img)

    print("Calibration method: click 2 points")
    print("Known length (m):", cfg.known_length_m)
    # Uncomment to run interactive calibration:
    # calib = estimate_scale_from_two_points(img, depth, K, cfg.known_length_m)
    # print(json.dumps(calib, indent=2))

    print("Note: calibration is interactive. Uncomment the call above when ready.")


## 11) Measurements

We compute (when possible):
- **Point-to-point distances** (3D)
- **Plane extents** (wall width/height estimates from plane inliers)
- **Door/window sizes** from masks projected into 3D
- **Areas** by projecting masks onto a plane and estimating surface area

All results are returned as a table and also exported to CSV with:
`object_type, label, width_m, height_m, depth_m, area_m2, location_xyz`.


In [ ]:
def mask_to_points(mask: np.ndarray, depth: np.ndarray, K: Intrinsics, depth_scale: float, max_points: int = 80_000) -> np.ndarray:
    ys, xs = np.where(mask)
    if ys.size == 0:
        return np.zeros((0,3), dtype=np.float32)
    if ys.size > max_points:
        idx = np.random.choice(ys.size, size=max_points, replace=False)
        ys, xs = ys[idx], xs[idx]
    z = depth[ys, xs].astype(np.float32) * float(depth_scale)
    x = (xs.astype(np.float32) - K.cx) * z / K.fx
    y = (ys.astype(np.float32) - K.cy) * z / K.fy
    pts = np.stack([x,y,z], axis=-1)
    mask_valid = np.isfinite(pts).all(axis=1) & (pts[:,2] > 0)
    return pts[mask_valid]

def extent_xy(pts: np.ndarray) -> Tuple[float,float,float]:
    """Returns width,height,depth extents in meters in camera coords."""
    if pts.shape[0] == 0:
        return 0.0, 0.0, 0.0
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    ext = maxs - mins
    return float(ext[0]), float(ext[1]), float(ext[2])

def centroid(pts: np.ndarray) -> Tuple[float,float,float]:
    if pts.shape[0] == 0:
        return (0.0,0.0,0.0)
    c = pts.mean(axis=0)
    return (float(c[0]), float(c[1]), float(c[2]))

def estimate_area_on_plane(pts: np.ndarray, plane_model: Optional[Tuple[float,float,float,float]] = None) -> float:
    """Approximate area by projecting to 2D via PCA and taking convex hull area.

    This is a heuristic; for better results, triangulate the surface.
    """
    if pts.shape[0] < 100:
        return 0.0
    X = pts - pts.mean(axis=0, keepdims=True)
    # PCA -> 2D
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    basis2 = Vt[:2].T  # 3x2
    xy = X @ basis2  # Nx2

    # Convex hull area
    try:
        from scipy.spatial import ConvexHull
        hull = ConvexHull(xy)
        return float(hull.volume)  # for 2D, 'volume' is area
    except Exception:
        return 0.0

def measure_scene_single(img_rgb: np.ndarray, depth: np.ndarray, masks: Dict[str,np.ndarray] | None, instances: List[Dict] | None, K: Intrinsics, depth_scale: float) -> pd.DataFrame:
    rows = []

    # Structural masks
    if masks is not None:
        for obj_type in ["wall","floor","ceiling","door","window","railing","stairs"]:
            m = masks.get(obj_type)
            if m is None or m.sum() < 200:
                continue
            pts = mask_to_points(m, depth, K, depth_scale)
            w,h,d = extent_xy(pts)
            area = estimate_area_on_plane(pts)
            rows.append({
                "object_type": obj_type,
                "label": obj_type,
                "width_m": w,
                "height_m": h,
                "depth_m": d,
                "area_m2": area,
                "location_xyz": centroid(pts),
            })

    # Furniture-ish instances
    if instances is not None:
        for inst in instances:
            pts = mask_to_points(inst["mask"], depth, K, depth_scale)
            w,h,d = extent_xy(pts)
            area = estimate_area_on_plane(pts)
            rows.append({
                "object_type": "object",
                "label": inst["label"],
                "width_m": w,
                "height_m": h,
                "depth_m": d,
                "area_m2": area,
                "location_xyz": centroid(pts),
            })

    df = pd.DataFrame(rows, columns=["object_type","label","width_m","height_m","depth_m","area_m2","location_xyz"])
    return df


## 12) Visualization + export

- Annotated image (labels)
- Open3D viewer (point cloud + planes)
- CSV export + PLY export
- End-of-run JSON-like summary


In [ ]:
def draw_labels(img_rgb: np.ndarray, df: pd.DataFrame) -> np.ndarray:
    out = img_rgb.copy()
    # We don't have exact pixel locations for each object; we annotate a legend block.
    x0, y0 = 10, 20
    font = cv2.FONT_HERSHEY_SIMPLEX
    for i, row in df.head(15).iterrows():
        text = f"{row['label']}: W={row['width_m']:.2f}m H={row['height_m']:.2f}m A={row['area_m2']:.2f}m2"
        cv2.putText(out, text, (x0, y0), font, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(out, text, (x0, y0), font, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        y0 += 18
    return out

def export_outputs(img_rgb: np.ndarray, pcd, df: pd.DataFrame, out_dir: str | Path):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    annotated = draw_labels(img_rgb, df)
    cv2.imwrite(str(out_dir / "annotated.png"), cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

    csv_path = out_dir / "measurements.csv"
    df2 = df.copy()
    df2["location_xyz"] = df2["location_xyz"].apply(lambda t: json.dumps(list(t)))
    df2.to_csv(csv_path, index=False)

    if o3d is not None and pcd is not None:
        o3d.io.write_point_cloud(str(out_dir / "pointcloud.ply"), pcd)

    print("Saved:")
    print(" -", out_dir / "annotated.png")
    print(" -", csv_path)
    if o3d is not None and pcd is not None:
        print(" -", out_dir / "pointcloud.ply")


def summary_json(df: pd.DataFrame) -> Dict:
    items = []
    for _, r in df.iterrows():
        items.append({
            "object_type": r["object_type"],
            "label": r["label"],
            "width_m": float(r["width_m"]),
            "height_m": float(r["height_m"]),
            "depth_m": float(r["depth_m"]),
            "area_m2": float(r["area_m2"]),
            "location_xyz": list(r["location_xyz"]),
        })
    return {"units": "meters", "count": len(items), "items": items}


## 13) Single-image mode — end-to-end demo (runs on `sample_data/`)

This is the primary Python-only pipeline:

A. Read input image
B. Depth estimation (Depth Anything V2)
C. Intrinsics + point cloud
D. Semantic masks (walls/door/window/etc.) + optional Mask R-CNN instances
E. Optional plane fitting
F. Metric calibration (interactive; you can skip to see relative units)
G. Measurements + export


In [ ]:
# Choose an image (sample_data/ or images/test.jpeg)
img_path = cfg.input_path
img = read_single_image(img_path)
show_img(img, f"Input: {img_path}")

# Depth
depth = infer_depth_da2(img, cfg.depth_input_size)
show_img(depth_to_vis(depth), "Depth (relative)")

# Intrinsics
K = get_intrinsics(img)
print("Intrinsics:", K)

# Semantic / instances
masks = semseg_masks(img) if cfg.use_semantic_segmentation else None
instances = maskrcnn_instances(img) if (cfg.use_maskrcnn_instances and cfg.mode == "single") else None

if masks is not None:
    show_img(overlay_masks(img, masks), "Semantic overlay")

if instances:
    print("Instances:", [(x["label"], round(x["score"], 2)) for x in instances])

# Calibration
# If you want metric outputs: uncomment interactive calibration below.
# calib = estimate_scale_from_two_points(img, depth, K, cfg.known_length_m)
# depth_scale = calib["scale_to_m"]
# print("Calibration:", json.dumps(calib, indent=2))

# If you skip calibration, depth_scale=1.0 gives relative units (NOT meters).
depth_scale = 1.0

# Point cloud
pcd = None
if o3d is not None:
    pcd = depth_to_pointcloud_o3d(img, depth, K, depth_scale=depth_scale, stride=3)
    print("Point cloud points:", np.asarray(pcd.points).shape)

# Planes (optional)
planes = []
if o3d is not None and pcd is not None:
    planes, rest = fit_planes_ransac(pcd, max_planes=4, distance_threshold=0.03)
    print("Planes found:", len(planes))
    for i, pl in enumerate(planes):
        t = classify_plane(pl["plane_model"])
        ext = plane_extent(pl["pcd_in"])
        print(f"  plane {i}: type={t}, extent≈{[round(x,3) for x in ext['extent']]}")

# 2D heuristics for mouldings
mould = detect_moulding_baseboard_heuristic(img, wall_mask=(masks.get("wall") if masks else None))
show_img(overlay_masks(img, {"baseboard": mould["baseboard"], "railing": mould["railing_like"], "crown": mould["crown_moulding"]}), "Heuristic moulding/baseboard/rail overlay")

# Measurements
df = measure_scene_single(img, depth, masks, instances, K, depth_scale)
display(df) if 'display' in globals() else print(df)

# Export
out_dir = ROOT / "outputs_single"
export_outputs(img, pcd, df, out_dir)

# JSON-like summary
summary = summary_json(df)
print(json.dumps(summary, indent=2)[:4000])


## 14) Multi-view mode (COLMAP SfM/MVS) — CLI cell + Python loader

COLMAP is not a Python library in most setups; we run it via shell commands.

### Install COLMAP
- macOS: `brew install colmap`
- Ubuntu: `sudo apt-get install -y colmap`

### Workflow
A. Put your photos in a folder (e.g. `./my_scene/`)
B. Run COLMAP to create a sparse model, then dense reconstruction
C. Load the resulting point cloud (`dense/fused.ply`) into Open3D
D. Apply scale (known object length or baseline) if needed

If `colmap` is not installed, skip this section.


In [ ]:
import shutil

def run_cmd(cmd: list[str], cwd: str | Path | None = None):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

# Configure your multi-view input folder:
COLMAP_IMAGE_DIR = Path(cfg.input_folder)
COLMAP_WORKDIR = ROOT / "outputs_colmap"

print("colmap available:", bool(shutil.which("colmap")))
print("input images:", COLMAP_IMAGE_DIR)

# ---- COLMAP commands (uncomment to run locally) ----
# NOTE: These can take minutes-hours.
#
# if not shutil.which("colmap"):
#     raise RuntimeError("COLMAP not found. Install it first.")
#
# COLMAP_WORKDIR.mkdir(parents=True, exist_ok=True)
# db = COLMAP_WORKDIR / "database.db"
# sparse = COLMAP_WORKDIR / "sparse"
# dense = COLMAP_WORKDIR / "dense"
# sparse.mkdir(exist_ok=True)
# dense.mkdir(exist_ok=True)
#
# run_cmd(["colmap", "feature_extractor", "--database_path", str(db), "--image_path", str(COLMAP_IMAGE_DIR)])
# run_cmd(["colmap", "exhaustive_matcher", "--database_path", str(db)])
# run_cmd(["colmap", "mapper", "--database_path", str(db), "--image_path", str(COLMAP_IMAGE_DIR), "--output_path", str(sparse)])
#
# # Pick the largest sparse model folder (often '0')
# sparse_model = sparse / "0"
#
# run_cmd(["colmap", "image_undistorter", "--image_path", str(COLMAP_IMAGE_DIR), "--input_path", str(sparse_model), "--output_path", str(dense), "--output_type", "COLMAP"])
# run_cmd(["colmap", "patch_match_stereo", "--workspace_path", str(dense), "--workspace_format", "COLMAP", "--PatchMatchStereo.geom_consistency", "true"])
# run_cmd(["colmap", "stereo_fusion", "--workspace_path", str(dense), "--workspace_format", "COLMAP", "--input_type", "geometric", "--output_path", str(dense / "fused.ply")])
#
# print("Dense point cloud:", dense / "fused.ply")

# ---- Loading dense point cloud (if exists) ----
FUSED_PLY = COLMAP_WORKDIR / "dense" / "fused.ply"
if o3d is not None and FUSED_PLY.exists():
    pcd_mv = o3d.io.read_point_cloud(str(FUSED_PLY))
    print("Loaded:", FUSED_PLY, "points:", np.asarray(pcd_mv.points).shape)
    # o3d.visualization.draw_geometries([pcd_mv])
else:
    print("No fused point cloud found yet. Run COLMAP commands above to generate it.")


## 15) RGB‑D / SLAM mode (RTAB‑Map / ORB‑SLAM3) — input + TSDF fusion (Open3D)

If you have RGB-D frames (metric depth), we can fuse them into a mesh/point cloud in Python with Open3D.

### Expected folder layout (example)
```
my_rgbd/
  rgb/
    000001.png
    ...
  depth/
    000001.png   # depth in meters or millimeters (configure scale)
  intrinsics.json
```

For **ORB‑SLAM3 / RTAB‑Map**, install externally; this notebook includes a command cell template.


In [ ]:
def load_intrinsics_json(path: str | Path) -> Intrinsics:
    data = json.loads(Path(path).read_text())
    return Intrinsics(fx=float(data["fx"]), fy=float(data["fy"]), cx=float(data["cx"]), cy=float(data["cy"]))

def fuse_rgbd_tsdf(rgb_dir: str | Path, depth_dir: str | Path, K: Intrinsics, depth_scale_to_m: float = 0.001, voxel_length: float = 0.02, sdf_trunc: float = 0.05, max_frames: int = 150):
    """TSDF fusion (requires metric depth).

    depth_scale_to_m converts raw depth units to meters:
      - if depth png is in millimeters: depth_scale_to_m = 0.001
      - if already meters: depth_scale_to_m = 1.0
    """
    if o3d is None:
        raise RuntimeError("Open3D not available")

    rgb_dir = Path(rgb_dir)
    depth_dir = Path(depth_dir)
    rgbs = sorted([p for p in rgb_dir.glob("*") if p.suffix.lower() in IMAGE_EXTS])
    depths = sorted([p for p in depth_dir.glob("*") if p.suffix.lower() in IMAGE_EXTS])
    n = min(len(rgbs), len(depths), max_frames)
    if n == 0:
        raise FileNotFoundError("No RGB-D frames found")

    intrinsic = o3d.camera.PinholeCameraIntrinsic()
    # Open3D expects width/height; we infer from first rgb
    rgb0 = cv2.imread(str(rgbs[0]))
    h, w = rgb0.shape[:2]
    intrinsic.set_intrinsics(w, h, K.fx, K.fy, K.cx, K.cy)

    volume = o3d.pipelines.integration.ScalableTSDFVolume(
        voxel_length=voxel_length,
        sdf_trunc=sdf_trunc,
        color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8,
    )

    # NOTE: Without poses (from SLAM/odometry), TSDF fusion can't align frames.
    # This function assumes identity pose for all frames unless you provide poses.
    # For real fusion, supply camera poses from SLAM (RTAB-Map / ORB-SLAM3) and integrate each frame with its pose.

    for i in range(n):
        rgb = o3d.io.read_image(str(rgbs[i]))
        dep = o3d.io.read_image(str(depths[i]))
        rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
            rgb,
            dep,
            depth_scale=1.0 / depth_scale_to_m,
            depth_trunc=10.0,
            convert_rgb_to_intensity=False,
        )
        T = np.eye(4)
        volume.integrate(rgbd, intrinsic, np.linalg.inv(T))

    mesh = volume.extract_triangle_mesh()
    mesh.compute_vertex_normals()
    pcd = volume.extract_point_cloud()
    return mesh, pcd

print("RGB-D/SLAM section ready. Provide RGB-D frames + poses for real fusion.")
print("RTAB-Map:", "https://github.com/introlab/rtabmap")
print("ORB-SLAM3:", "https://github.com/UZ-SLAMLab/ORB_SLAM3")


## 16) Troubleshooting

- **No masks / wrong labels**: semantic models may not label your domain well. Try a different segmentation checkpoint.
- **Depth looks inverted**: Depth Anything V2 outputs a relative depth map; visualization normalizes it. For geometry, scale and sign matter—ensure depth values are positive.
- **Point cloud is warped**: intrinsics are wrong. Provide correct `fx/fy/cx/cy` (camera calibration).
- **Measurements in “relative units”**: you skipped calibration in single-image mode. Use the two-point known-length calibration.
- **Open3D import errors**: install `open3d` compatible with your Python version; consider Python 3.10–3.12.
- **Detectron2 install issues**: use the default `transformers` semantic segmentation path instead.
- **COLMAP not found**: install via brew/apt or skip multi-view.

### Practical capture tips
- For single-image: include a reference object (door, ruler, A4 paper) and avoid extreme wide-angle distortion.
- For multi-view: take overlapping photos with texture and varied viewpoints; avoid reflective/transparent surfaces.
- For RGB-D: prefer sensor-provided depth + known intrinsics.


## 17) Small unit-test style checks

These are lightweight sanity checks to catch obvious calibration/intrinsics issues.


In [ ]:
def _assert_close(a, b, tol, msg):
    if abs(a - b) > tol:
        raise AssertionError(f"{msg}: {a} vs {b} (tol={tol})")

# Synthetic test: if depth is constant and we pick two pixels separated by dx,
# the 3D distance in x should scale linearly with depth.

h, w = 100, 200
img_dummy = np.zeros((h, w, 3), dtype=np.uint8)
K_test = Intrinsics(fx=100.0, fy=100.0, cx=w/2, cy=h/2)

u1, v1 = 50.0, 50.0
u2, v2 = 60.0, 50.0
z = 2.0
X1 = backproject_pixel(u1, v1, z, K_test)
X2 = backproject_pixel(u2, v2, z, K_test)

# Expected: dx3D = (du * z / fx)
expected = (u2 - u1) * z / K_test.fx
observed = float(np.linalg.norm(X2 - X1))
_assert_close(observed, expected, tol=1e-5, msg="Backprojection distance")

# Scale test
known_m = 1.5
scale = known_m / observed
_assert_close(scale * observed, known_m, tol=1e-6, msg="Scale application")

print("Sanity checks passed.")
